<a href="https://colab.research.google.com/github/ketan49-coder/ECG-SNN-FPGA/blob/main/SNN_SuperSpike_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install snntorch wfdb seaborn -q
!git clone https://github.com/ketan49-coder/ECG-SNN-FPGA.git
%cd ECG-SNN-FPGA/python

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 15.1 MB/s eta 0:00:00
Cloning into 'ECG-SNN-FPGA'...
remote: Enumerating objects: 818, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 818 (delta 22), reused 45 (delta 13), pack-reused 762 (from 2)
Receiving objects: 100% (818/818), 78.21 MiB | 19.63 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/ECG-SNN-FPGA/python


In [ ]:
import wfdb, numpy as np, os
from sklearn.model_selection import train_test_split

# Fallback local directory mapped to the actual location in the cloned repository
DRIVE_DIR = "/content/drive/MyDrive/mit-bih-arrhythmia-database-1.0.0/mit-bih-arrhythmia-database-1.0.0"
LOCAL_DIR = "/content/ECG-SNN-FPGA/mit-bih-arrhythmia-database-1.0.0/mit-bih-arrhythmia-database-1.0.0"

if os.path.exists(DRIVE_DIR):
    DATASET_DIR = DRIVE_DIR
    print(f"Using dataset from Google Drive: {DATASET_DIR}")
else:
    DATASET_DIR = LOCAL_DIR
    print(f"Using local database directory: {DATASET_DIR}")

os.makedirs("data", exist_ok=True)
AAMI = {'N':0,'L':0,'R':0,'e':0,'j':0,'A':1,'a':1,'S':1,'J':1,'V':2,'E':2,'F':3,'/':4,'f':4,'Q':4}
RECORDS = ['100','101','102','103','104','105','106','107','108','109','111','112','113','114','115','116','117','118','119','121','122','123','124','200','201','202','203','205','207','208','209','210','212','213','214','215','217','219','220','221','222','223','228','230','231','232','233','234']

X_all, y_all = [], []
print("Extracting records from existing local files...")
for rec in RECORDS:
    p = os.path.join(DATASET_DIR, rec)
    # Only process if header and data files actually exist locally to avoid failing
    if os.path.exists(p + ".hea") and os.path.exists(p + ".dat"):
        try:
            sig = wfdb.rdrecord(p, channels=[0]).p_signal.flatten()
            ann = wfdb.rdann(p, 'atr')
            for peak, sym in zip(ann.sample, ann.symbol):
                if sym in AAMI:
                    l, r = peak-64, peak+64
                    if l>=0 and r<len(sig):
                        w = sig[l:r]
                        mn, mx = w.min(), w.max()
                        if mx > mn:
                            X_all.append((w-mn)/(mx-mn))
                            y_all.append(AAMI[sym])
        except Exception as e:
            pass

if len(X_all) == 0:
    raise ValueError("No samples extracted! Verify that .hea, .dat, and .atr files are fully present in your folder.")

X = np.array(X_all, dtype=np.float32)
y = np.array(y_all, dtype=np.int64)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
np.save("data/X_train.npy", X_tr)
np.save("data/X_test.npy",  X_te)
np.save("data/y_train.npy", y_tr)
np.save("data/y_test.npy",  y_te)
print(f"Data extraction complete! Loaded {len(X_all)} samples. ✅")

Using local database directory: /content/ECG-SNN-FPGA/mit-bih-arrhythmia-database-1.0.0/mit-bih-arrhythmia-database-1.0.0
Extracting records from existing local files...
Data extraction complete! Loaded 109480 samples. ✅


In [ ]:
%%writefile train_snn_superspike.py
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import snntorch as snn
from snntorch import surrogate
from torch.utils.data import DataLoader, TensorDataset

NUM_STEPS = 25
OUTPUT_CLASSES = 5
BATCH_SIZE = 128
MODEL_DIR = "/content/drive/MyDrive/SNN_Models"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class WeightedVanRossumLoss(nn.Module):
    def __init__(self, class_weights, tau=2.0):
        super().__init__()
        self.decay = np.exp(-1.0 / tau)
        self.register_buffer('class_weights', torch.tensor(class_weights, dtype=torch.float32))

    def forward(self, spk_out, targets):
        T, B, C = spk_out.shape
        target_spikes = torch.zeros(T, B, C, device=spk_out.device)
        for b in range(B):
            target_spikes[:, b, targets[b]] = 1.0

        filtered_out, filtered_tgt = torch.zeros_like(spk_out), torch.zeros_like(target_spikes)
        trace_out, trace_tgt = torch.zeros(B, C, device=spk_out.device), torch.zeros(B, C, device=spk_out.device)
        for t in range(T):
            trace_out = trace_out * self.decay + spk_out[t]
            trace_tgt = trace_tgt * self.decay + target_spikes[t]
            filtered_out[t] = trace_out
            filtered_tgt[t] = trace_tgt

        # Apply class-specific weights to the MSE loss
        mse = F.mse_loss(filtered_out, filtered_tgt, reduction='none') # (T, B, C)
        weighted_mse = mse * self.class_weights.view(1, 1, C)
        return weighted_mse.sum(dim=(0, 2)).mean()

class ECG_SuperSpike(nn.Module):
    def __init__(self):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid(slope=25)
        self.fc1 = nn.Linear(128, 32, bias=False)
        self.fc2 = nn.Linear(32, 32, bias=False)
        self.fc3 = nn.Linear(32, OUTPUT_CLASSES, bias=False)
        self.lif1 = snn.Leaky(beta=0.9375, threshold=0.5, spike_grad=spike_grad, reset_mechanism="zero")
        self.lif2 = snn.Leaky(beta=0.9375, threshold=0.5, spike_grad=spike_grad, reset_mechanism="zero")
        self.lif3 = snn.Leaky(beta=0.9375, threshold=0.5, spike_grad=spike_grad, reset_mechanism="zero")
        self.drop = nn.Dropout(p=0.3)

    def forward(self, x):
        mem1, mem2, mem3 = self.lif1.init_leaky(), self.lif2.init_leaky(), self.lif3.init_leaky()
        spk3_rec = []
        drop_mask = torch.rand(NUM_STEPS) > 0.1 if self.training else torch.ones(NUM_STEPS, dtype=torch.bool)
        for t in range(NUM_STEPS):
            if not drop_mask[t]:
                spk3_rec.append(torch.zeros(x.size(0), OUTPUT_CLASSES, device=x.device))
                continue
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(self.drop(spk1))
            spk2, mem2 = self.lif2(cur2, mem2)
            cur3 = self.fc3(spk2)
            spk3, mem3 = self.lif3(cur3, mem3)
            spk3_rec.append(spk3)
        return torch.stack(spk3_rec, dim=0)

@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    t_ok, t_tot = 0, 0
    class_ok = [0]*5
    class_tot = [0]*5

    for data, targets in loader:
        preds = net(data.to(device)).sum(0).argmax(1).cpu()
        targets_cpu = targets.cpu()
        t_ok += (preds == targets_cpu).sum().item()
        t_tot += targets.size(0)
        for c in range(5):
            class_ok[c] += ((preds == c) & (targets_cpu == c)).sum().item()
            class_tot[c] += (targets_cpu == c).sum().item()

    class_names = ['Normal (N)', 'Supraventricular (S)', 'Ventricular (V)', 'Fusion (F)', 'Unknown (Q)']
    print("\n     Per-class accuracy:")
    for c in range(5):
        acc = (100.0 * class_ok[c] / class_tot[c]) if class_tot[c] > 0 else 0.0
        bar = "█" * int(acc / 5)
        print(f"       {class_names[c]:<30} {acc:>6.2f}%  {bar}")
    print()
    return 100.0 * t_ok / t_tot

def train_model():
    X_train, y_train = np.load("data/X_train.npy"), np.load("data/y_train.npy")
    X_test, y_test = np.load("data/X_test.npy"), np.load("data/y_test.npy")

    # Calculate inverse frequency class weights to balance the loss
    class_counts = np.bincount(y_train, minlength=5)
    total_samples = len(y_train)
    class_weights = total_samples / (5.0 * class_counts + 1e-5)
    # Normalize weights
    class_weights = class_weights / np.sum(class_weights) * 5.0
    print("Using Balanced Class Weights:", class_weights)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=BATCH_SIZE, shuffle=False)

    net = ECG_SuperSpike().to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=2e-3)
    loss_fn = WeightedVanRossumLoss(class_weights=class_weights).to(device)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-5)
    os.makedirs(MODEL_DIR, exist_ok=True)

    best_acc = 0.0
    for epoch in range(40):
        net.train()
        total_loss = 0.0
        for data, targets in train_loader:
            spk_out = net(data.to(device))
            loss = loss_fn(spk_out, targets.to(device))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        test_acc = evaluate(net, test_loader)
        print(f"Epoch {epoch+1}/40 | Loss: {total_loss/len(train_loader):.4f} | Test Acc: {test_acc:.2f}% | LR: {scheduler.get_last_lr()[0]:.2e}")
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(net.state_dict(), f"{MODEL_DIR}/snn_superspike_best.pth")

if __name__ == "__main__":
    train_model()

Overwriting train_snn_superspike.py


In [ ]:
!python train_snn_superspike.py

Using device: cuda
Using Balanced Class Weights: [0.02936079 0.95663406 0.36774547 3.31543736 0.33082232]

     Per-class accuracy:
       Normal (N)                      99.88%  ███████████████████
       Supraventricular (S)             0.00%  
       Ventricular (V)                 43.81%  ████████
       Fusion (F)                       0.00%  
       Unknown (Q)                     85.27%  █████████████████

Epoch 1/40 | Loss: 11.4007 | Test Acc: 91.83% | LR: 2.00e-03

     Per-class accuracy:
       Normal (N)                      99.71%  ███████████████████
       Supraventricular (S)             0.00%  
       Ventricular (V)                 65.72%  █████████████
       Fusion (F)                       0.00%  
       Unknown (Q)                     88.69%  █████████████████

Epoch 2/40 | Loss: 9.3187 | Test Acc: 93.40% | LR: 1.99e-03

     Per-class accuracy:
       Normal (N)                      99.64%  ███████████████████
       Supraventricular (S)             0.00%  
     